# IOAI — 2025 Gaite Contest Word Segmentation (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/IOAI-official/IOAI-2025"
CLONE = "IOAI-2025"
SUBDIR = "GAITE-Contest/Word_Segmentation"
WORKDIR = "GAITE-Contest/Word_Segmentation"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, WORKDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

## The reference answer to this question by Scientific Committee is rated 0.90

In [ ]:
import random
import numpy as np
import torch

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## ⚠️ 제약 (반드시 지킬 것) — 예측 JSON 제출형 (독일어 합성어 분할 · 글자별 경계)

베이스라인을 **자유롭게 개선**해 `submission.zip` 을 만드세요. 모델 구조·학습은 자유(제공 데이터만 사용). 단:

1. **작업 영역 = `## Train phase` 의 `class MyModel`** — 베이스라인(one-hot+MLP=0점)을 **Embedding + LSTM/BiLSTM** 으로 개선(권장).
2. **학습 + 추론 모두 포함 필수** — 학습 과정을 생략하고 가중치만 내는 건 **불가**. (모델/`.pth`/마운트 데이터 제출 불가, **노트북만** 제출)
3. **출력**: `## Validation and Test phase`·`## Submission Format` 셀이 `val.json`/`test.json` 을 추론해 **`submissionval.json` + `submissiontest.json`** 생성 → `submission.zip`. 형식은 training 과 동일한 JSON `{단어: [0/1 배열]}` (글자 위치별 **1=단어 끝, 0=시작/중간**).
4. **인터넷 없음**(pip/conda/API 불가, 오프라인 pretrained 가능), **20분 실행 제한**, 50회 제출 제한.

**채점**: **단어별 F1 평균** (분할 위치가 정확할수록 높음). val=Leaderboard A(공개), test=B(비공개).

## Train phase

In [ ]:
# The reference answer to this question (Scientific Committee) is rated 0.95
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import logging
import zipfile
import os

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

device = 'cuda'

with open("train.json", "r") as f:
    data = list(json.load(f).items())

# Character vocabulary
chars = sorted(list(set("".join([word for word, _ in data]))))
char2idx = {char: idx + 1 for idx, char in enumerate(chars)}  # 0 is reserved for padding
idx2char = {idx: char for char, idx in char2idx.items()}
vocab_size = len(chars)

# Define Dataset
class CompoundDataset(Dataset):
    def __init__(self, data, char2idx):
        self.data = data
        self.char2idx = char2idx

    def __len__(self):
        return len(self.data)

    def encode(self, word, labels):
        return (
            torch.tensor([self.char2idx[char] for char in word], dtype=torch.long),
            torch.tensor(labels, dtype=torch.float),
        )

    def __getitem__(self, idx):
        word, labels = self.data[idx]
        return self.encode(word, labels)


# Collate function to handle batching
def collate_fn(batch):
    inputs, targets = zip(*batch)
    lengths = [len(seq) for seq in inputs]
    max_len = max(lengths)

    padded_inputs = torch.zeros(len(inputs), max_len, dtype=torch.long)
    padded_targets = torch.zeros(len(targets), max_len, dtype=torch.float)

    for i, (seq, tgt) in enumerate(zip(inputs, targets)):
        padded_inputs[i, : len(seq)] = seq
        padded_targets[i, : len(tgt)] = tgt

    return padded_inputs, padded_targets, lengths

# Define the pure MLP Model (no embeddings)  
# Using a pure neural network, the runtime score will most likely be 0
class MyModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim=128):
        super(MyModel, self).__init__()
        self.vocab_size = vocab_size
        
        # Input size is vocab_size (one-hot dimension)
        self.fc1 = nn.Linear(vocab_size + 1, hidden_dim)  # +1 for padding index
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()
        self.relu = nn.ReLU()

    def forward(self, x):
       
        # x: (batch_size, seq_length)
        
        # Convert to one-hot encoding
        x_onehot = torch.zeros(x.size(0), x.size(1), self.vocab_size + 1).to(x.device)
        x_onehot.scatter_(2, x.unsqueeze(-1), 1)
        
        # Process each position independently with MLP
        x = self.relu(self.fc1(x_onehot))
        x = self.relu(self.fc2(x))
        logits = self.fc_out(x).squeeze(-1)  # (batch_size, seq_length)
        return self.sigmoid(logits)

def train():
    # Initialize Dataset and DataLoader
    dataset = CompoundDataset(data, char2idx)
    
    batch_size = 128
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, num_workers=4, prefetch_factor=2
    )
    
    # Initialize Model, Loss, Optimizer
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MyModel(vocab_size).to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # Training Loop
    num_epochs = 8
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        for inputs, targets, lengths in dataloader:
            targets = targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs.to(device))
            
            # Mask padding positions
            mask = torch.arange(inputs.shape[1])[None, :] < torch.tensor(lengths)[:, None]
            mask = mask.to(device)
            outputs = outputs[mask]
            targets = targets[mask]
            
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        logging.info(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss/len(dataloader):.4f}")
    return model

model = train()

## Validation and Test phase

In [ ]:
def predict_and_save(model, input_file, output_file, char2idx, device="cpu"):
    """
    Reads a JSON file, predicts segmentation for each word, and saves the results to a new JSON file.

    :param model: Trained model
    :param input_file: Path to input JSON file
    :param output_file: Path to output JSON file
    :param char2idx: Character to index mapping
    :param device: Device to run the model on
    """
    # Load the input data
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Initialize predictions dictionary
    predictions = {}

    # Set model to evaluation mode
    model.eval()

    # Predict for each word
    with torch.no_grad():
        for word, _ in data.items():
            # Convert word to indices
            indices = [char2idx.get(char, 0) for char in word]
            input_tensor = torch.tensor(indices, dtype=torch.long).unsqueeze(0).to(device)
            
            # Get model outputs
            outputs = model(input_tensor)[0].cpu().numpy()
            
            # Convert outputs to binary labels
            boundaries = (outputs > 0.6).astype(int)
            predictions[word] = boundaries.tolist()

    # Save predictions to output file
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=4)

    logging.info(f"Predictions saved to {output_file}")


## Submission Format
When the baseline is running, this error message will appear because the test set cannot be read through DATA_PATH on testing machine, which is a normal phenomenon.

In [ ]:
#DATA_PATH is the secret environment variable to point the address of the validation set and test set on the testing machine. 
#You cannot access this address locally.
if os.environ.get('DATA_PATH'):
    data_path = os.environ.get("DATA_PATH") + "/"  
else:
    data_path = ""
    print("Colab: 동봉된 검증/테스트 입력을 사용합니다(정상).") #Colab: 동봉된 검증/테스트 입력을 사용합니다(정상).
# Predict and save results
input_file = data_path + "val.json"
output_file = "./submissionval.json"
predict_and_save(model, input_file, output_file, char2idx, device)
# Predict and save results
input_file = data_path + "test.json"
output_file = "./submissiontest.json"
predict_and_save(model, input_file, output_file, char2idx, device)
with zipfile.ZipFile('submission.zip', 'w') as zipf:
    zipf.write('submissionval.json')
    zipf.write('submissiontest.json')

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submissionval.json', 'submissiontest.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)